<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Building Single Agents with the OpenAI Agents SDK
## Overview
In this demo, you will build a single Data Analyst agent from scratch using the OpenAI Agents SDK on Databricks. You will define a custom tool with `@function_tool`, create and test the agent, register tools to Unity Catalog for governance, and enable MLflow tracing to observe execution.

## Learning Objectives
By the end of this demo, you will be able to:
1. Configure the OpenAI Agents SDK to use Databricks model routing
2. Define custom tools using `@function_tool`
3. Build and test a single agent with tools
4. Register tools to Unity Catalog for governance
5. Enable MLflow autologging and inspect execution traces

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;">This demo assumes you have completed notebooks 01-03 (Foundations, UC Functions Demo, and Fix a Bad Agent Tool Lab) and are familiar with <strong>Unity Catalog</strong> functions and <strong>MCP</strong> tool discovery.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

In [0]:
%run ./Includes/Classroom-Setup-5

### A1. Import Libraries

To get started with the OpenAI SDK, we have a few things that need to be setup:
1. Configure the OpenAI Agents SDK to use Databricks model routing.
1. Import MCP components needed to connect to agent tools that have already been registered to UC.
1. Initialize the Databricks `WorkspaceClient`.
1. Set the default trace processor to MLflow.

Additional documentation and reading:
- [`set_default_openai_client`](https://openai.github.io/openai-agents-python/ref/#agents.set_default_openai_client)
- [`set_default_openai_api`](https://openai.github.io/openai-agents-python/ref/#agents.set_default_openai_api)
- [`set_trace_processors`](https://github.com/openai/openai-agents-python/blob/main/docs/tracing.md)

In [0]:
from agents import Agent, Runner, function_tool, set_default_openai_api, set_default_openai_client
from agents.tracing import set_trace_processors
from databricks_openai import AsyncDatabricksOpenAI
import logging

set_default_openai_client(AsyncDatabricksOpenAI())
set_default_openai_api("chat_completions")
set_trace_processors([])  # Disable OpenAI trace processor; use MLflow instead
logging.getLogger("openai").setLevel(logging.WARNING)

print("OpenAI Agents SDK configured to use Databricks model routing.")

## B. Define the Data Analyst Agent

### B1. Define the `classify_price_tier` Tool

The `classify_price_tier` tool classifies an Airbnb listing's nightly price into one of four market tiers based on the SF pricing distribution:

| Tier | Price Range | Percentile Band |
|---|---|---|
| **Budget** | Under $100 | Bottom 25% |
| **Mid-Range** | $100 -- $174 | 25--50% |
| **Premium** | $175 -- $299 | 50--75% |
| **Luxury** | $300+ | Top 25% |

The `@function_tool` decorator registers this function as a tool that the agent can invoke. The SDK extracts the function signature and docstring to generate the tool schema that the LLM uses to decide when and how to call the tool.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
An example production workflow is: prototype with <code>@function_tool</code>, then register to Unity Catalog, then access via a Managed MCP server. This demo follows the first two steps of that progression. In Section C, we explicitly re-register the function to Unity Catalog for governance.
</p>
</div>
</div>
</div>

In [0]:
import json

@function_tool
def classify_price_tier(price: float) -> str:
    """Classify an Airbnb listing price into a market tier based on SF pricing distribution.

    Uses hardcoded SF Airbnb market thresholds to label a nightly price as
    Budget, Mid-Range, Premium, or Luxury, and returns the corresponding
    percentile band for context.

    Useful for questions like: 'Is $180/night a good deal in SF?',
    'What tier does this listing fall into?'

    Returns a JSON object with tier, percentile_band, and a short interpretation.

    Args:
        price: Nightly listing price in USD.
    """
    import json

    tiers = [
        (100,  "Budget",    "bottom 25%",  "Well below the SF market average."),
        (175,  "Mid-Range", "25–50%",      "Around the SF market median."),
        (300,  "Premium",   "50–75%",      "Above average for SF listings."),
        (float("inf"), "Luxury", "top 25%", "Among the most expensive listings in SF."),
    ]

    if price < 0:
        return json.dumps({"error": "Price must be a positive value."})

    for threshold, tier, band, interpretation in tiers:
        if price < threshold:
            return json.dumps({
                "price": price,
                "tier": tier,
                "percentile_band": band,
                "interpretation": interpretation
            }, indent=2)

print(f"Defined tool: {classify_price_tier.name}")

### B2. Create the Agent

With the tool defined, we create a single agent that uses `classify_price_tier` to answer questions about SF Airbnb pricing. The `instructions` parameter shapes the agent's behavior and tells it when to use the tool. 

This is accomplished with the `Agent` class. An agent is an AI model configured with instructions, tools, guardrails, etc. In this notebook we will focus creating a simple agent by passing a `name`, set of `instructions` (which acts as the system prompt), a foundation model serving endpoint via `model`, and `tools`. You can read more about `Agent` [here](https://github.com/openai/openai-agents-python/blob/main/src/agents/agent.py).

In [0]:
data_analyst_agent = Agent(
    name="Data Analyst",
    instructions="You are a data analyst that helps users understand SF Airbnb listing prices. Use the classify_price_tier tool to categorize prices into market tiers. Provide clear, concise answers.",
    model="databricks-gpt-5-2",
    tools=[classify_price_tier],
)

print(f"Created agent: {data_analyst_agent.name}")
print(f"  Model: {data_analyst_agent.model}")
print(f"  Tools: {[t.name for t in data_analyst_agent.tools]}")

### B3. Test the Agent

We send three progressively complex queries to validate that the agent correctly invokes the tool and interprets the results. Each call uses `await Runner.run()`, which is the execution interface. It executes the full agent loop: the LLM decides whether to call a tool, executes it if needed, and generates a final response.

1. The next cell will run a straightforward pricing question that requires a single tool call.

In [0]:
query_1 = "Is $150/night a good deal in SF?"
result_1 = await Runner.run(data_analyst_agent, query_1)
print(f"User:     {query_1}")
print(f"Response: {result_1.final_output}")

2. The next cell will run a direct tier classification at the high end of the price spectrum.

In [0]:
query_2 = "What tier would a $500/night luxury listing fall into?"
result_2 = await Runner.run(data_analyst_agent, query_2)
print(f"User:     {query_2}")
print(f"Response: {result_2.final_output}")

3. The next cell will run multiple prices in a single request, testing whether the agent calls the tool multiple times.

In [0]:
query_3 = "Classify these prices: $80, $200, $350"
result_3 = await Runner.run(data_analyst_agent, query_3)
print(f"User:     {query_3}")
print(f"Response: {result_3.final_output}")

## C. Register Tools to Unity Catalog

### C1. Register `classify_price_tier` as a Python UC Function

The inline `@function_tool` is local to this notebook session. To make the tool governed, discoverable, and reusable across agents and workspaces, we register it to Unity Catalog as a Python function using `DatabricksFunctionClient`.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
The function registered below is a <strong>plain Python function</strong> (no <code>@function_tool</code> decorator). The <code>DatabricksFunctionClient</code> inspects the function signature and docstring to create the UC function metadata. The logic is identical to the tool we prototyped above.
</p>
</div>
</div>
</div>

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

def classify_price_uc(price: float) -> str:
    """Classify an Airbnb listing price into a market tier based on SF pricing distribution.

    Uses hardcoded SF Airbnb market thresholds to label a nightly price as
    Budget, Mid-Range, Premium, or Luxury, and returns the corresponding
    percentile band for context.

    Useful for questions like: 'Is $180/night a good deal in SF?',
    'What tier does this listing fall into?'

    Returns a JSON object with tier, percentile_band, and a short interpretation.

    Args:
        price: Nightly listing price in USD.
    """
    import json

    tiers = [
        (100,  "Budget",    "bottom 25%",  "Well below the SF market average."),
        (175,  "Mid-Range", "25–50%",      "Around the SF market median."),
        (300,  "Premium",   "50–75%",      "Above average for SF listings."),
        (float("inf"), "Luxury", "top 25%", "Among the most expensive listings in SF."),
    ]

    if price < 0:
        return json.dumps({"error": "Price must be a positive value."})

    for threshold, tier, band, interpretation in tiers:
        if price < threshold:
            return json.dumps({
                "price": price,
                "tier": tier,
                "percentile_band": band,
                "interpretation": interpretation
            }, indent=2)

client = DatabricksFunctionClient()
function_info = client.create_python_function(
    func=classify_price_uc,
    catalog=catalog_name,
    schema=schema_name,
    replace=True,
)

print(f"Created UC function: {function_info.full_name}")

### C2. Verify Registered Functions

Confirm that the function is visible in the Unity Catalog schema, then call it directly to verify it returns the expected output.
1. Navigate to the **Catalog Explorer** on the left side menu and search for `classify_price_uc`
1. Run the next cell to test the output from calling the function from UC

In [0]:
# Call classify_price_tier directly to verify
result = spark.sql(f"SELECT {catalog_name}.{schema_name}.classify_price_uc(150.0) AS result")
display(result)

### C3. Access the UC Function via a Managed MCP Server

Databricks provides **Managed MCP servers** that automatically expose Unity Catalog functions as tools. Any agent can discover and call these functions through the MCP protocol without custom integration code.

The MCP server endpoint for UC functions follows this pattern:
`https://workspace/api/2.0/mcp/functions/catalog/schema`

With the OpenAI Agents SDK, you use `McpServer` from `databricks_openai.agents` to connect as shown in the next cell.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
The <code>McpServer</code> class from <code>databricks_openai.agents</code> is an async context manager designed for the Databricks Apps runtime. In a notebook, we use <code>DatabricksMCPClient</code> from <code>databricks_mcp</code> to interact with the same MCP endpoint.
See: Managed MCP servers (<a href="https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">GCP</a>)
</p>
</div>
</div>
</div>


In [0]:
import nest_asyncio
nest_asyncio.apply()

from databricks_mcp import DatabricksMCPClient
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
workspace_host = ws.config.host

# Point to the MCP server for our catalog/schema
mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=ws)

# Discover available tools (nest_asyncio patches the event loop for DatabricksMCPClient)
tools = mcp_client.list_tools()
print(f"Discovered {len(tools)} UC function tool(s):")
for t in tools:
    print(f"  - {t.name}: {t.description}")

### C4. Call the UC Function Through MCP

Now we call the registered UC function through the MCP protocol, just as an agent would. The default tool name is of the form `catalogName__schemaName__toolName` as demonstrated in the next cell.

In [0]:
# Call the UC function via MCP
tool_name = f"{catalog_name}__{schema_name}__classify_price_uc"
result = mcp_client.call_tool(tool_name, {"price": 120.00})
print(f"Result: {''.join([c.text for c in result.content])}")

## D. Enable MLflow Tracing

### D1. Enable Autologging

MLflow autologging for the OpenAI integration automatically captures execution traces for every agent run. Each trace records the full sequence of LLM calls, tool invocations, inputs, outputs, and latencies without requiring any code changes to the agent itself.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
Autologging captures: LLM call spans (model, prompt, completion tokens), tool call spans (function name, arguments, return value), and the overall agent run span with total latency. All traces are stored in the MLflow experiment associated with this notebook.
</p>
</div>
</div>
</div>

In [0]:
import mlflow

mlflow.openai.autolog()

print("MLflow OpenAI autologging enabled.")
print(f"Traces will appear in experiment: {mlflow_experiment_path}")

### D2. Tracing without MCP

With autologging enabled, the next agent run will produce a full execution trace. The trace captures the LLM reasoning step, the tool call to `classify_price_tier`, and the final response generation.

In [0]:
traced_query = "I found a listing for $225/night in the Mission district. Is that a fair price?"
result = await Runner.run(data_analyst_agent, traced_query)
print(f"User:     {traced_query}")
print(f"Response: {result.final_output}")

### D3. Tracing with MCP

In a Databricks App deployment, the `McpServer` from `databricks_openai.agents` is used as an async context manager with the `Agent()` class.

<div style="display:flex;align-items:center;gap:0;margin:20px 0;flex-wrap:wrap;font-family:system-ui,sans-serif;">
  <div style="background:#F9F7F4;border:2px solid #1B5162;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">@function_tool<br/><span style="font-weight:400;font-size:12px;">Local only</span></div>
  <div style="color:#999;font-size:22px;padding:0 8px;">&#x2192;</div>
  <div style="background:#2574B5;color:#fff;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">UC Function<br/><span style="font-weight:400;font-size:12px;">Governed + reusable</span></div>
  <div style="color:#999;font-size:22px;padding:0 8px;">&#x2192;</div>
  <div style="background:#02A36F;color:#fff;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">MCP Server<br/><span style="font-weight:400;font-size:12px;">Agent-accessible + standardized</span></div>
  <div style="color:#999;font-size:22px;padding:0 8px;">&#x2192;</div>
  <div style="background:#FF5F46;color:#fff;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">Agent<br/><span style="font-weight:400;font-size:12px;">Production-ready</span></div>
</div>

This is the recommended progression: start with `@function_tool` for rapid prototyping, register to UC for governance, then expose through MCP for production agent access.

1. Run the next cell to expose only `classify_price_uc` to the agent. 

In [0]:
from databricks_openai.agents import McpServer

mcp_server = McpServer.from_uc_function(
    catalog=catalog_name,
    schema=schema_name,
    function_name="classify_price_uc",
    timeout=60.0,
)

2. **Redefine our agent** from before, but this time we pass `mcp_servers=[mcp_server]` instead of `tools`. The agent now discovers its tools at runtime from the UC function MCP server rather than from inline `@function_tool` definitions.

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
Why do we need @mlflow.trace, async def, and async?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #333;">
<ul style="margin: 0; padding-left: 20px;">
<li><strong><code>@mlflow.trace(...)</code></strong>: Without this, the MCP connection, LLM call, and tool call each produce separate traces. Wrapping them in a single decorated function concatenates all spans into one unified trace.</li>
<li><strong><code>async def</code></strong>: <code>Runner.run()</code> is async (it makes HTTP calls to the model endpoint), and <code>McpServer</code> is an async context manager. Both require <code>await</code>, which means the enclosing function must be <code>async def</code>.</li>
<li><strong><code>async with mcp_server:</code></strong>: Opens the HTTP connection to the Databricks Managed MCP server before the agent runs, and closes it after. The agent calls <code>tools/list</code> and <code>tools/call</code> over this connection during execution.</li>
</ul>
</div>
</details>

In [0]:
data_analyst_mcp_agent_update = Agent(
    name="Data Analyst",
    instructions="You are a data analyst that helps users understand SF Airbnb listing prices. Use the classify_price_tier tool to categorize prices into market tiers. Provide clear, concise answers.",
    model="databricks-gpt-5-2",
    mcp_servers=[mcp_server], # <---- used to be tools exposed via @function_tool
)

# Traced wrapper that concatenates all spans into a single trace
@mlflow.trace(span_type="AGENT", name="data_analyst_mcp_execution")
async def run_data_analyst_mcp(question: str) -> str:
    """Run the Data Analyst agent with MCP tools as a single traced operation."""
    async with mcp_server:
        result = await Runner.run(data_analyst_mcp_agent_update, question)
        return result.final_output

In [0]:
response = await run_data_analyst_mcp(query_1)
print(f"User:     {query_1}")
print(f"Response: {response}")

## E. Conclusion

In this demo, you:

1. Configured the OpenAI Agents SDK to route requests through Databricks model serving
2. Defined a `classify_price_tier` tool using the `@function_tool` decorator to classify Airbnb prices into market tiers
3. Built a single Data Analyst agent and tested it with progressively complex pricing queries
4. Registered `classify_price_tier` as a Python UC function for governance and connected it to the agent via MCP
5. Enabled MLflow autologging and inspected execution traces capturing LLM calls, tool invocations, and response generation

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>